In [3]:
import json
from pathlib import Path

# Path to ORIGINAL FHIR dataset
FHIR_DIR = Path("../../Forge-N-FHIR/synthetic-ivy-hip-emr/FHIR")

er_encounter_count = 0
unique_patients = set()

for file in FHIR_DIR.rglob("*.json"):
    with open(file) as f:
        resource = json.load(f)

    if resource.get("resourceType") == "Encounter":
        # Check if encounter location is ER
        locations = resource.get("location", [])
        for loc in locations:
            ref = loc.get("location", {}).get("reference", "")
            if ref == "Location/er":
                er_encounter_count += 1

                # Get patient
                subject = resource.get("subject", {}).get("reference")
                if subject:
                    patient_id = subject.split("/")[-1]
                    unique_patients.add(patient_id)

print("ER Encounters:", er_encounter_count)
print("Unique Patients in ER:", len(unique_patients))

ER Encounters: 97
Unique Patients in ER: 59


In [7]:
import requests

url = "http://localhost:8080/fhir/Encounter"
params = {
    "location": "Location/er",
    "_count": 500
}

response = requests.get(url, params=params)
data = response.json()

encounters = data.get("entry", [])

unique_patients = set()

for entry in encounters:
    resource = entry["resource"]
    subject = resource.get("subject", {}).get("reference")
    if subject:
        patient_id = subject.split("/")[-1]
        unique_patients.add(patient_id)

print("ER encounters in HAPI:", len(encounters))
print("Unique patients in HAPI:", len(unique_patients))

ER encounters in HAPI: 93
Unique patients in HAPI: 58


4 encounters missing
1 patient missing
Identify Which 4 Are Missing
Get ER encounter IDs from local folder


In [10]:
# Collect Local ER Patient IDs
local_patients = set()

for file in FHIR_DIR.rglob("*.json"):
    with open(file) as f:
        resource = json.load(f)

    if resource.get("resourceType") == "Encounter":
        for loc in resource.get("location", []):
            ref = loc.get("location", {}).get("reference", "")
            if ref == "Location/er":
                subject = resource.get("subject", {}).get("reference")
                if subject:
                    local_patients.add(subject.split("/")[-1])

print("Local ER patients:", len(local_patients))

Local ER patients: 59


In [9]:
import requests

hapi_er_ids = set()

url = "http://localhost:8080/fhir/Encounter"
params = {
    "location": "Location/er",
    "_count": 500
}

response = requests.get(url, params=params)
data = response.json()

for entry in data.get("entry", []):
    resource = entry["resource"]
    hapi_er_ids.add(resource.get("id"))

print("HAPI ER encounters:", len(hapi_er_ids))

HAPI ER encounters: 93


In [11]:
# Collect ER Patient IDs from HAPI
import requests

hapi_patients = set()

url = "http://localhost:8080/fhir/Encounter"
params = {
    "location": "Location/er",
    "_count": 500
}

response = requests.get(url, params=params)
data = response.json()

for entry in data.get("entry", []):
    resource = entry["resource"]
    subject = resource.get("subject", {}).get("reference")
    if subject:
        hapi_patients.add(subject.split("/")[-1])

print("HAPI ER patients:", len(hapi_patients))

HAPI ER patients: 58


In [12]:
# Find the Missing Patient
missing_patient = local_patients - hapi_patients

print("Missing patient in HAPI:")
print(missing_patient)

Missing patient in HAPI:
{'patient-PKI7P'}


In [13]:
# ER encounter(s) that caused the discrepancy
for file in FHIR_DIR.rglob("*.json"):
    with open(file) as f:
        resource = json.load(f)

    if resource.get("resourceType") == "Encounter":
        subject = resource.get("subject", {}).get("reference")
        if subject and subject.split("/")[-1] in missing_patient:
            print(resource.get("id"))


encounter-2018_PKI7P-8ZN
encounter-2018_PKI7P-QK8
encounter-2018_PKI7P-NWE
encounter-2018_PKI7P-Z9C
encounter-2018_PKI7P-EFJ
encounter-2018_PKI7P-OCM
encounter-2018_PKI7P-0K3
